## Trabajo Práctico 1 - Tecnología Digital VI: Machine Learning
 
**Integrantes:** Lourdes Moreira, Juana Arslanian y Matías Cerdeira  

**Fecha de entrega:** 2 de septiembre de 2026

## Introducción

En este trabajo desarrollamos dos problemas de clasificación con el mismo conjunto de datos. La Parte I vamos a analizar el problema de abandono de clientes, conocido como *Customer Churn*, utilizando un dataset de una empresa de telecomunicaciones y poniendo el foco en la prevención de *data leakage*, la validación y las métricas de negocio. La Parte II reformula el objetivo para predecir si un cliente con servicio de internet contrataría o no servicio de *streaming*.

## Parte I

### 1. Problema y Análisis Exploratorio de los datos

Una empresa de telecomunicaciones quiere identificar clientes con riesgo de abandonar el servicio. Cada fila representa un cliente e incluye información sobre su antigüedad, los servicios que tiene contratados, el tipo de contrato, su forma de pago y sus cargos mensuales, entre otras variables. La variable que queremos predecir, es decir, el target `Churn` vale `Yes` cuando el cliente abandonó el servicio. 

Anticipar qué clientes tienen mayor probabilidad de irse permitiría dirigir acciones de retención, como ofrecer descuentos a tiempo. Sin embargo, el objetivo no es predecir muchos `No`, sino encontrar una proporción útil de las fugas sin enviar descuentos indiscriminadamente. Por eso más adelante contrastamos *precision* y *recall* y no evaluamos el modelo solo con *accuracy*.

De este modo, comenzamos cargando los datos y miramos las primeras 5 filas para ver qué variables incluye el dataset y obtener un panorama más claro a la hora de armar el modelo.

In [54]:
import pandas as pd

datos = pd.read_csv("dataset Telco Customer Churn.csv", na_values=[" "]) 
#aclaramos cómo tratar a los vacíos para que pandas pueda leer TotalCharges como columna númerica y no como texto
print("Filas:", datos.shape[0], "| Columnas:", datos.shape[1])


datos.head()

Filas: 7043 | Columnas: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


COSAASSS

No alcanza con obtener un modelo con una **accuracy** alta. Como la mayoría de los clientes no abandona el servicio, un modelo podría acertar muchas veces simplemente prediciendo siempre la clase mayoritaria y, aun así, no detectar ninguna fuga real.

Estas métricas permiten distinguir entre dos errores con consecuencias diferentes: ofrecer un descuento a alguien que no se iba y no detectar a un cliente que finalmente abandona.

Finalmente, vamos a utilizar el mismo dataset para plantear un segundo problema de clasificación con una variable objetivo diferente. Esto nos permitirá observar cómo un mismo conjunto de datos puede utilizarse para responder preguntas de negocio distintas.

un cliente con muchos cargos acumulados probablemente tenga una relación diferente con la empresa que uno recién ingresado.

Detectamos variables categóricas y numéricas, y consultamos ciertos features que hacen más fácil entender el dataset. Estos features son numéricos:
- `tenure`: Es la cantidad de meses que el cliente lleva en la empresa.
- `TotalCharges`: Es el total acumulado que se le cobró durante toda su relación con la empresa.

Luego, chequeamos si hay alguna columna que contenga datos faltantes, cuál es y cuántos posee, ya que puede contener información útil que no esté siendo bien procesada.

In [55]:
faltantes = datos.isna().sum()

faltantes[faltantes > 0]

TotalCharges    11
dtype: int64

*-*-* FALTA *-*-* 

#### Visualizaciones

Al cargar los datos, es posible que algunas columnas no sean reconocidas directamente
como num´ericas. Identifiquen por qu´e sucede esto, corrijan el problema y expliquen qu´e
impacto tendr´ıa en un modelo si este tipo de anomal´ıas no se detectan a tiempo:::

### 1.2 Anomalía de `TotalCharges` y estrategia de faltantes

`TotalCharges` debería ser numérica, pero pandas la carga como texto porque contiene 11 valores formados solo por espacios. 

Primero convertimos los espacios en `NaN` y luego usamos `pd.to_numeric`. La auditoría muestra además que los 11 casos tienen `tenure = 0`: son cuentas recién iniciadas, sin cargos acumulados. Por conocimiento del dominio asignamos `TotalCharges = 0`; no usamos una estadística aprendida con todo el dataset. Imputarles la mediana global introduciría un acumulado ficticio muy alto. El pipeline mantendrá igualmente un imputador entrenado dentro de cada fold como protección frente a futuros faltantes.

*-*-* FALTA *-*-* 

### 2. Preprocesamiento y prevención de Data Leakage

Como vimos que `TotalCharges` tiene 11 datos faltantes conformados por espacios en blanco `""`, observamos que corresponden a clientes con `tenure = 0` lo que significa que son nuevos y que no se les a cobrado nada aún. De este modo, decidimos imputar con 0 porque sino un modelo podría por ejemplo excluir silenciosamente una variable potencialmente informativa.

DATA LEAKAGE EXPLICARLO ???

In [56]:
datos["TotalCharges"] = datos["TotalCharges"].fillna(0)

#verificamos
print("Cantidad de datos faltantes en todo el dataset:", datos.isna().sum().sum())


Cantidad de datos faltantes en todo el dataset: 0


Ahora separamos las features del target y organizamos donde: 
- `X` contiene las features utilizadas para predecir,
- `y` contiene lo que queremos predecir.

Excluimos `customerID` porque es único por persona y sirve sólo de identificador por lo que no contiene una señal generalizable y permitirlo como predictor agregaría miles de categorías sin significado. También, convertimos `Churn` en 0 y 1 para que después sea más fácil calcular las métricas, en vez de `Yes`, `No`.

In [57]:
from sklearn.model_selection import train_test_split

X = datos.drop(columns=["customerID", "Churn"])

y = datos["Churn"].map({
    "No": 0,
    "Yes": 1
})

Luego, dividimos el dataset en conjunto de entrenamiento y evaluación, en el que el 80% corresponde a Train y el 20% a Test, usamos una semilla fija = 42 para que la división sea siempre la misma pues sino cada ejecución puede producir un train y un test diferentes, y conservamos aproximadamente la misma proporción de clientes que abandonan en train y test con ayuda de stratify.

In [58]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

#### Importante aclaración:

Nosotros simplemente limpiamos el dataset al corregir los espacios vacíos con 0, puesto que los valores faltantes correspondían a clientes con antigüedad igual a cero, por lo que se completaron directamente con cero ya que no utiliza información del conjunto de test, así que no genera data leakage. Como no utilizamos una estadística calculada sobre todo el dataset podíamos hacerlo antes del Train/Test Split. Sin embargo, si hubiéramos utilizado una media o mediana, habría sido necesario calcularla utilizando solamente el conjunto de entrenamiento, ya que sino el promedio incluiría también a los clientes que después quedarían en test.

#### Armado del modelo

Utilizamos una regresión logística porque `Churn` es una variable binaria, donde cada cliente pertenece a la clase que abandona o a la que no abandona.

Con `OneHotEncoder`, separamos variables numéricas y categóricas porque `LogisticRegression` no puede trabajar directamente con textos. Tras haber analizado el dataset, sabemos cuáles son las numéricas para discriminarlas y tratar todas las otras como categóricas. Asimismo, escalamos las numéricas para colocarlas en escalas comparables con `StandardScaler`. Esto es conveniente para una regresión logística porque utiliza regularización.

Tanto la transformación de las categorías como el escalado se colocan dentro de un pipeline. De esta manera se ajustan únicamente con los datos de entrenamiento. Si calculáramos el escalado con todo el dataset antes del split, estaríamos incorporando información del conjunto de test, lo que produciría data leakage.

In [59]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

variables_numericas = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
variables_categoricas = [
    columna
    for columna in X.columns
    if columna not in variables_numericas
]
preprocesador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            StandardScaler(),
            variables_numericas
        ),
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categoricas
        )
    ]
)
modelo = Pipeline(
    steps=[
        ("preprocesamiento", preprocesador),
        (
            "regresion_logistica",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

### 3. Estrategia de validación

Como una partición *holdout* evalúa el modelo una sola vez, el resultado puede depender de qué clientes cayeron por casualidad en validación (o evaluación). Para evitar esto, *K-Fold* con K = 5 folds sobre el conjunto de entrenamiento, permite que en cada iteración el modelo se entrene con cuatro folds y se valide con el restante. El proceso se repite hasta que cada fold haya sido utilizado una vez para validación y así cada parte del conjunto de entrenamiento se utiliza una vez para validar.

Esta estrategia permite obtener cinco mediciones diferentes en lugar de depender de una única partición *holdout*. De esta manera, podemos calcular un desempeño promedio y observar si los resultados cambian demasiado según los datos utilizados para validar y una estimación más estable.

El procedimiento se repite cinco veces, de modo que cada observación participa una vez en la validación.
K-Fold permite obtener una evaluación más estable que una única partición holdout, ya que el resultado no depende de una sola división de los datos. Además, podemos calcular el promedio y la variación de las métricas entre los cinco folds.
En este dataset sería conveniente utilizar StratifiedKFold porque la variable Churn está desbalanceada. La clase de clientes que no abandona representa aproximadamente el 73 %, mientras que la clase que abandona representa cerca del 27 %. A diferencia de K-Fold común, la versión estratificada conserva aproximadamente esta proporción en cada fold, haciendo que las evaluaciones sean más comparables.

Elegimos StratifiedKFold porque la variable Churn está desbalanceada: la cantidad de clientes que no abandona es considerablemente mayor que la cantidad que sí abandona. La estratificación mantiene aproximadamente la misma proporción de ambas clases en cada fold y hace que las métricas sean más comparables.
La validación cruzada se realiza únicamente sobre el conjunto de entrenamiento. El conjunto de test permanece separado y se utilizará una sola vez para evaluar el modelo final. Además, incluimos el preprocesamiento dentro del pipeline para evitar que la imputación, el escalado o la codificación aprendan información del fold utilizado para validación.

In [60]:
from sklearn.model_selection import KFold, cross_validate

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
resultados = cross_validate(
    modelo,
    X_train,
    y_train,
    cv=kfold,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1"
    ]
)
resultados_cv = pd.DataFrame({
    "Accuracy": resultados["test_accuracy"],
    "Precision": resultados["test_precision"],
    "Recall": resultados["test_recall"],
    "F1": resultados["test_f1"]
})

resultados_cv.round(3)

,Accuracy,Precision,Recall,F1
0,0.802,0.633,0.538,0.582
1,0.821,0.645,0.589,0.616
2,0.800,0.697,0.525,0.599
3,0.807,0.659,0.578,0.616
4,0.787,0.641,0.518,0.573


In [61]:
resultados_cv.mean().round(3)

Accuracy     0.804
Precision    0.655
Recall       0.550
F1           0.597
dtype: float64

El conjunto de test queda guardado para la evaluación final.
En cada vuelta se entrena el pipeline completo:
1. Se ajusta el escalado con cuatro folds.
2. Se aprende el One-Hot Encoding con cuatro folds.
3. Se entrena la regresión logística.
4. Se evalúa con el fold restante.

LAS METRICAS para q ahora???

#### Stratified K-Fold

Creemos que usar Stratified K-Fold sería conveniente ya que el problema del K-Fold común es que no garantiza que cada fold tenga la misma proporción de clientes que abandonan. Entonces para conjunto de datos **desbalanceados** como este, donde aproximadamente el 73% no abandona y el 27% sí, resulta buena estrategia utilizar `StratifiedKFold` que intenta mantener esa proporción en cada fold.

De este modo, el modelo que utilizamos se implementa así

In [62]:
##Con stratified K folds?? que se espera?
#validacion cruzada puede ser cualquier cosa osea k o stratified?

La validación cruzada se utilizó para evaluar el comportamiento del modelo en cinco particiones diferentes del conjunto de entrenamiento. Una vez observados esos resultados, entrenamos el modelo final utilizando todo el conjunto de entrenamiento disponible. Finalmente, lo evaluamos sobre el conjunto de test, que se había mantenido separado.

### 4. Evaluación del modelo base

Entrenamos el modelo con todo el conjunto de entrenamiento y hacemos predicciones en las que para cada cliente de test guardamos 1 si predice que abandona, y 0 si no.

In [63]:
modelo.fit(X_train, y_train);

predicciones = modelo.predict(X_test)

Calculamos métricas para ver qué tan bueno resultó el modelo y las organizamos en una tabla.

In [64]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
accuracy = accuracy_score(y_test, predicciones)
precision = precision_score(y_test, predicciones)
recall = recall_score(y_test, predicciones)
f1 = f1_score(y_test, predicciones)
metricas = pd.DataFrame({
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1-Score": [f1]
})

metricas.round(3)

,Accuracy,Precision,Recall,F1-Score
0,0.806,0.657,0.559,0.604


#### ¿Qué representa cada métrica?

- **Accuracy** es el porcentaje total de predicciones correctas
- **Precision** se fija de todos los clientes que el modelo señaló como posibles fugas, indica qué porcentaje realmente abandonó. En este caso, con precision de 0.657 significa que alrededor del 66% de los clientes señalados por el modelo efectivamente abandonó.
- **Recall** mira de todos los clientes que realmente abandonaron, indica qué porcentaje logró detectar el modelo. Acá el recall de 0.559 significa que el modelo detectó cerca del 56% de las fugas reales.
- **F1-Score** combina precision y recall en una sola métrica. Es útil cuando queremos considerar ambos tipos de error.

#### Matriz de confusión

Ahora calculamos la matriz de confusión en la que se incluyen los éxitos, fracaso, falsos positivos y falsos negativos.

In [65]:
from sklearn.metrics import confusion_matrix

matriz = confusion_matrix(y_test, predicciones)
matriz_confusion = pd.DataFrame(
    matriz,
    index=["Real: No abandona", "Real: Abandona"],
    columns=["Predice: No abandona", "Predice: Abandona"]
)

matriz_confusion

,Predice: No abandona,Predice: Abandona
Real: No abandona,926,109
Real: Abandona,165,209


COMO VEO Q PORCENTAJES TIENE DE LOS 2 PARA DETERMINAR CHURN=0

In [66]:
clase_mayoritaria = (y == 0).mean() * 100

round(clase_mayoritaria, 1)

np.float64(73.5)

#### Modelo naive

Un modelo *naive* que predice la clase mayoritaria tendría un **accuracy** del 73,5% el cual parece muy alto pero es solamente porque la mayoría de los clientes no abandona. Como el accuracy se calcula haciendo:
$$
\frac{Predicciones~correctas}{Total~de~casos}
$$
acertaría siempre que sea `Churn = 0`.

Sin embargo, este modelo no detectaría ningún cliente que abandona .su **recall** para la clase positiva sería cero pues siempre predice `No`.

Esto muestra que la **accuracy** puede resultar engañosa cuando las clases están desbalanceadas y que debe complementarse con métricas como precision, recall y F1-Score. De este modo, es una buena métrica pero no como única métrica.

En este problema, *precision* indica qué proporción de los descuentos enviados llega realmente a personas que se iban, *recall* indica qué proporción de las fugas logramos contactar y F1 resume el equilibrio entre ambas.

Una breve demo

In [67]:
import numpy as np
predicciones_naive = np.zeros(len(y_test))

accuracy_naive = accuracy_score(
    y_test,
    predicciones_naive
)

round(accuracy_naive, 3)

0.735

### 5. Análisis de Negocio e Impacto de las Métricas

Creamos un segundo modelo aplicando la técnica de **Random Oversampling** sobre el conjunto de entrenamiento para lidiar con el desbalanceo de clases. En este caso, tenemos 4139 clientes que no abandonan y 1495 que sí. Para ello, creamos una tabla de entrenamiento que incluya el target y copiamos los datos de X y completamos a la clase minoritaria para que tenga la misma cantidad de observaciones, luego de hacer el Train/Test Split, porque si lo hiciéramos antes un mismo cliente podría terminar repetido tanto en train como en test, lo que sería *data leakage*. Por ende, no estamos creando clientes completamente nuevos sino repitiendo aleatoriamente observaciones de clientes que sí abandonaron. El conjunto de test no se modifica ya que debe conservar la distribución original para representar una evaluación realista.

La desventaja es que el oversampling repite observaciones existentes y puede aumentar el riesgo de sobreajuste, pero por ejemplo hacer **Undersampling** descartaría casi la mitad de la información de entrenamiento. De todas formas, resulta una técnica sencilla y adecuada para analizar cómo cambia el comportamiento del modelo cuando ambas clases tienen el mismo peso durante el entrenamiento.

In [68]:
y_train.value_counts()

Churn
0    4139
1    1495
Name: count, dtype: int64

In [69]:
from sklearn.utils import resample
from sklearn.base import clone

entrenamiento = X_train.copy()

entrenamiento["Churn"] = y_train

clase_mayoritaria = entrenamiento[
    entrenamiento["Churn"] == 0
]

clase_minoritaria = entrenamiento[
    entrenamiento["Churn"] == 1
]
clase_minoritaria_ampliada = resample(
    clase_minoritaria,
    replace=True,
    n_samples=len(clase_mayoritaria),
    random_state=42
)
entrenamiento_over = pd.concat([
    clase_mayoritaria,
    clase_minoritaria_ampliada
])
#para duplicar filas completas las habíamos unido pero las volvemos a separar para entrenar luego
X_train_over = entrenamiento_over.drop(
    columns="Churn"
)
y_train_over = entrenamiento_over["Churn"]
#verificamos el nuevo balance
y_train_over.value_counts()

Churn
0    4139
1    4139
Name: count, dtype: int64

Ahora que ambas clases tienen la misma cantidad de observaciones, creamos el nuevo modelo, lo entrenamos y evaluamos.

In [70]:
modelo_oversampling = clone(modelo)

modelo_oversampling.fit(
    X_train_over,
    y_train_over
);
predicciones_over = modelo_oversampling.predict(
    X_test
)

#### Métricas nuevas y comparación

Calculamos las métricas y comparamos con el modelo base.

In [71]:
accuracy_over = accuracy_score(
    y_test,
    predicciones_over
)

precision_over = precision_score(
    y_test,
    predicciones_over
)

recall_over = recall_score(
    y_test,
    predicciones_over
)

f1_over = f1_score(
    y_test,
    predicciones_over
)

comparacion_modelos = pd.DataFrame({
    "Modelo": [
        "Modelo base",
        "Modelo con oversampling"
    ],
    "Accuracy": [
        accuracy,
        accuracy_over
    ],
    "Precision": [
        precision,
        precision_over
    ],
    "Recall": [
        recall,
        recall_over
    ],
    "F1-Score": [
        f1,
        f1_over
    ]
})

comparacion_modelos.round(3)

,Modelo,Accuracy,Precision,Recall,F1-Score
0,Modelo base,0.806,0.657,0.559,0.604
1,Modelo con oversampling,0.733,0.498,0.781,0.608


Construimos la matriz de confusión

In [72]:
matriz_over = confusion_matrix(
    y_test,
    predicciones_over
)

matriz_over_df = pd.DataFrame(
    matriz_over,
    index=[
        "Real: No abandona",
        "Real: Abandona"
    ],
    columns=[
        "Predice: No abandona",
        "Predice: Abandona"
    ]
)

matriz_over_df

,Predice: No abandona,Predice: Abandona
Real: No abandona,741,294
Real: Abandona,82,292


Observamos que el **Oversampling**:
- Reduce los falsos negativos de 165 a 82.
- Aumenta los verdaderos positivos de 209 a 292.
- Detecta 83 fugas adicionales.
- Aumenta los falsos positivos de 109 a 294.
- Genera 185 falsos positivos adicionales.
Esto explica por qué aumenta el recall pero disminuyen precision y accuracy.

#### Interpretación de falso positivo

Un falso positivo es un cliente que el modelo identifica como posible fuga, pero que en realidad no iba a abandonar. La empresa le ofrecería un descuento innecesario del 50 % durante tres meses y perdería parte de la facturación que habría recibido de todas formas.

#### Interpretación de Falso negativo
Un falso negativo es un cliente que realmente abandona, pero que el modelo no identifica. La empresa pierde la oportunidad de ofrecerle una acción de retención y puede perder los ingresos futuros asociados con ese cliente.

#### Si el descuento es muy costoso
Si el descuento fuera muy costoso, intentaríamos maximizar precision para evitar ofrecerlo a demasiados clientes que no iban a abandonar. El modelo base tiene una precision aproximada de 0,657 y genera 109 falsos positivos, mientras que el modelo con oversampling tiene una precision cercana a 0,498 y genera 294 falsos positivos. En este escenario sería preferible el modelo base.

#### Si el descuento es casi gratuito
Si el descuento fuera casi gratuito, intentaríamos maximizar recall para detectar la mayor cantidad posible de fugas. El modelo con oversampling alcanza un recall aproximado de 0,781 y reduce los falsos negativos de 165 a 82. En este escenario sería preferible el modelo con oversampling, aunque contacte a más clientes que finalmente no iban a abandonar.

# Parte II: Reformulando el Problema

Existen variables que al negocio no le interesaría predecir porque ya las conoce o porque como `tenure` y `TotalCharges` se conocen después de que el cliente ya eligió su contrato.

De este modo, elegimos predecir `StreamingMovies` dado que al negocio podría serle útil para hacer una campaña de cross-selling, es decir, ofrecer el servicio de películas a clientes de internet que todavía no lo tienen.

Como sus campos son `Yes`, `No`, `No internet service`, aquellos que no tienen servicio contratado no pueden contratar streaming, por eso es necesario excluirlos ya que no tendría sentido que el modelo aprenda que alguien sin internet no tiene streaming, porque sería una predicción demasiado obvia.

Transformamos el target en una variable binaria: asignamos `1` a los clientes que tienen StreamingMovies y `0` a quienes tienen internet, pero no contrataron el servicio.

Eliminamos las variables que no sirven y aquellas que podrían revelar indirectamente la respuesta como `MonthlyCharges` y `TotalCharges`, y dejamos variables que pueden describir el perfil del cliente sin contener directamente el precio del target. Claramente excluimos `StreamingMovies` porque sino habría *data leakage* pues el modelo recibiría la respuesta que queremos predecir.

In [73]:
#conservamos solamente clientes que tienen internet
datos_streaming = datos[
    datos["InternetService"] != "No" #Excluye???
].copy()

#nuevo target: 1 tiene StreamingMovies, 0 no tiene
y_streaming = (
    datos_streaming["StreamingMovies"] == "Yes"
).astype(int)

#features del nuevo problema sacando los que no van
X_streaming = datos_streaming.drop(
    columns=[
        "customerID",
        "Churn",
        "StreamingMovies",
        "MonthlyCharges",
        "TotalCharges"
    ]
)

#nueva división de entrenamiento y test
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_streaming,
    y_streaming,
    test_size=0.20,
    random_state=42,
    stratify=y_streaming
)

Creamos un train y un test nuevos porque ahora tenemos otro target

no queda desbalanceado!

Separamos variables categoricas y numericas pero como eliminamos los cargos, la única variable verdaderamente numérica es `tenure`, por lo que las demás son todas categóricas. 
También, realizamos el preprocesamiento como ya habíamos hecho antes donde escalamos `tenure` y convertimos las variables de texto en columnas numéricas.

In [74]:
numericas_streaming = [
    "tenure"
]
categoricas_streaming = [
    columna
    for columna in X_streaming.columns
    if columna not in numericas_streaming
]

preprocesador_streaming = ColumnTransformer(
    transformers=[
        (
            "numericas",
            StandardScaler(),
            numericas_streaming
        ),
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categoricas_streaming
        )
    ]
)


### Armado del nuevo modelo que predice StreamingMovies

Este modelo sigue siendo *out-of-the-box* porque usamos una regresión logística simple, no hacemos búsqueda de hiperparámetros, no seleccionamos el modelo según test y limitamos la cantidad de iteraciones máximas del entrenamiento a 1000.

In [75]:
modelo_streaming = Pipeline(
    steps=[
        (
            "preprocesamiento",
            preprocesador_streaming
        ),
        (
            "regresion_logistica",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

modelo_streaming.fit(
    Xs_train,
    ys_train
);

predicciones_streaming = modelo_streaming.predict(
    Xs_test
)

Ahora calculamos las métricas de testeo

In [76]:
accuracy_streaming = accuracy_score(
    ys_test,
    predicciones_streaming
)

precision_streaming = precision_score(
    ys_test,
    predicciones_streaming
)

recall_streaming = recall_score(
    ys_test,
    predicciones_streaming
)

f1_streaming = f1_score(
    ys_test,
    predicciones_streaming
)

Mostramos los resultados

In [77]:
metricas_streaming = pd.DataFrame({
    "Accuracy": [accuracy_streaming],
    "Precision": [precision_streaming],
    "Recall": [recall_streaming],
    "F1-Score": [f1_streaming]
})

metricas_streaming.round(3)

,Accuracy,Precision,Recall,F1-Score
0,0.728,0.728,0.72,0.724


En este caso, nos damos cuenta que un modelo naive en el que siempre se predice la clase mayoritaria claramente sería peor pues este conjunto no está desbalanceado y el accuracy quedaría de 0,505 lo que es lo peor que podría pasar pues equivale a tirar una moneda, y el modelo basado en una regresión logística tiene 0,728 de accuracy.

Entrenamos una regresión logística simple, sin búsqueda de hiperparámetros. El modelo obtuvo una accuracy aproximada de 0,728 y un F1-Score cercano a 0,724. Como el modelo naive alcanza solamente alrededor de 0,505 de accuracy, concluimos que el nuevo objetivo es razonablemente predecible.

# Conclusiones

1. **Calidad de datos:** `TotalCharges` se cargaba como texto por 11 espacios en blanco. Todos correspondían a cuentas con antigüedad cero, por lo que se corrigieron a un acumulado de 0. No había filas ni identificadores duplicados.
2. **Prevención de leakage:** el split se realizó antes de cualquier transformación aprendida. Imputación defensiva, escalado y one-hot quedaron dentro del pipeline y, por lo tanto, también dentro de cada fold de validación.
3. **Validación:** Stratified 5-Fold produjo una estimación más estable que un único holdout y conservó la proporción de churn en todos los folds. Test se usó una sola vez al final.
4. **Modelo base:** mejora claramente al clasificador mayoritario, pero la accuracy por sí sola oculta que la clase positiva es minoritaria. Precision, recall, F1 y la matriz de confusión describen mejor la utilidad.
5. **Desbalance y negocio:** ponderar clases aumenta el recall y reduce falsos negativos, a costa de más falsos positivos, menor precision y mayor gasto en descuentos. El modelo preferible depende del costo de la oferta y del valor esperado de retener a un cliente.
6. **Nuevo problema:** el contrato mensual es predecible con las variables restantes y supera ampliamente su baseline. La interpretación es asociativa porque el dataset no aporta una dimensión temporal.

## Mejoras futuras

- estimar el efecto causal de la campaña mediante un experimento A/B; un buen clasificador de churn no garantiza que el descuento cambie la decisión;
- incorporar margen, costo de contacto, probabilidad de aceptación y valor de vida del cliente para optimizar un umbral económico;
- comparar modelos no lineales y calibrar probabilidades mediante validación cruzada, sin ajustar decisiones sobre test;
- validar en un período posterior para detectar deriva y medir generalización temporal;
- auditar desempeño por subgrupos y excluir atributos sensibles cuando no sean necesarios para la decisión.
